# 09 — Production Pipeline: Redis Streams + Worker + SSE

Through notebook 08b we have an agent that's **stateful and interruptible**. Now we make it run **as a worker behind a queue**.

**Why this matters.** Notebook 07b's `app.ainvoke()` is fine for a notebook. But production agents need:
- The user can close their browser; the agent keeps running
- N workers can consume the same queue (horizontal scale)
- A worker crash redelivers — and **must not re-bill** a task that already completed
- The UI wants a live progress bar (SSE), not a 60-second blank screen

**By the end of this notebook you will:**
1. Enqueue tasks to a Redis Stream and watch a worker consume them
2. See the **SETNX idempotency lock** prevent double-execution under simulated redelivery
3. Stream progress events via Redis Pub/Sub as the agent runs
4. Verify the **six-step ordering** from S9 §5.4 — consume → check → lock → run → store → ack
5. (Optional) Wire in the real LangGraph brief from notebook 07b for end-to-end

Lecture reference: **S9 Part 5** (entire chapter maps to this notebook).


## 1. Start Redis

This notebook needs a local Redis. Three options:

```bash
# Option A — Docker (recommended, isolated)
docker run -d --name deepbrief-redis -p 6379:6379 redis:7-alpine

# Option B — Homebrew on macOS
brew install redis && brew services start redis

# Option C — apt on Debian / Ubuntu
sudo apt install redis-server && sudo systemctl start redis
```

Verify with `redis-cli ping` → `PONG`.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from redis.asyncio import Redis

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379/0")
redis = await Redis.from_url(REDIS_URL)

# Confirm we can talk to Redis
pong = await redis.ping()
print(f"Redis @ {REDIS_URL}: {'OK' if pong else 'NO'}")
assert pong, "Redis not reachable — start it (see the markdown cell above)"

Clean any leftover state from previous runs of this notebook so the demos are deterministic:

In [ ]:
from deepbrief.pipeline.streams import STREAM

# Wipe out the stream + any old consumer groups so the demo starts clean
await redis.delete(STREAM)
# Lock keys + run records
for key in await redis.keys("agent:lock:*"):
    await redis.delete(key)

import os, pathlib
db = pathlib.Path("./data/runs.db")
if db.exists():
    db.unlink()
print("clean state — ready to demo")

## 2. The pipeline shape

Five small files in `src/deepbrief/pipeline/`. Each maps to one box in the lecture diagram:

```
┌──────────────────┐  enqueue()      ┌─────────────────┐  consume_loop()  ┌────────────────┐
│ Producer         │ ──────────────▶ │ Queue           │ ───────────────▶ │ Worker         │
│ FastAPI POST /runs│                │ Redis Stream    │                  │ 6-step loop    │
└──────────────────┘                 │ `agent:runs`    │                  │                │
                                     └─────────────────┘                  └────────────────┘
                                                                                 │
                                                            store.insert_one()   ▼
                                                            ┌─────────────┐
                                                            │ Result store│
                                                            │  (SQLite)   │
                                                            └─────────────┘
                                                                 ▲
                                  progress.emit()   ┌─────────────┴───────────────┐
                                                    │ Redis Pub/Sub               │
                                                    │ `agent:progress:{task_id}`  │
                                                    └─────────────────────────────┘
                                                                 │
                                                          SSE GET /runs/{id}/stream
```

| File | What it does | Maps to |
|---|---|---|
| `streams.py` | `enqueue` + `consume_loop` over Redis Streams | §5.1 |
| `locks.py` | SETNX idempotency lock + Lua release | §5.2 |
| `progress.py` | Pub/Sub `ProgressEmitter` + `subscribe()` SSE source | §5.3 |
| `store.py` | SQLite result store (drop-in for Mongo) | §5.4 step 5 |
| `worker.py` | The six-step `make_handler` wiring all of the above | §5.4 |
| `api.py` | FastAPI app — POST /runs, GET /runs/{id}/stream | §5.1 producer surface |


## 3. Read the lock — the canonical Redis pattern

The lock is the **most important** piece. Get this wrong and you get either silent data loss or double-billing. The lecture spends a whole section (§5.2) on why the release uses a Lua script.

In [ ]:
import inspect
from deepbrief.pipeline.locks import IdempotencyLock, _RELEASE_SCRIPT
print(inspect.getsource(IdempotencyLock))
print()
print("_RELEASE_SCRIPT (Lua, runs atomically in Redis):")
print(_RELEASE_SCRIPT)

**Why is the release a Lua script?** Three steps in plain Python:

```python
# WRONG — race condition window between GET and DEL
current = await redis.get(lock_key)
if current == worker_id:
    await redis.delete(lock_key)
```

Between the `GET` and the `DEL`, the lock can expire and a different worker can claim it. Your `DEL` then deletes *their* lock. The Lua script bundles GET+DEL into one atomic operation Redis executes without interruption.

**Senior interview tip:** when asked about distributed locks, *mention the Lua script*. It's the single piece of evidence that you've shipped this pattern instead of read about it.

## 4. Test the lock in isolation

Two workers race for the same task. One wins, one loses. Each releases only its own lock.

In [ ]:
lock = IdempotencyLock(redis, ttl_seconds=10)

won_a = await lock.acquire("task-001", worker_id="worker-A")
won_b = await lock.acquire("task-001", worker_id="worker-B")
print(f"worker-A acquire: {won_a}")
print(f"worker-B acquire: {won_b}")
assert won_a and not won_b

# worker-B tries to release a lock it doesn't own — Lua script blocks this
released_b = await lock.release("task-001", worker_id="worker-B")
print(f"worker-B release (NOT owner): {released_b}")

released_a = await lock.release("task-001", worker_id="worker-A")
print(f"worker-A release (owner):     {released_a}")
assert not released_b and released_a

**This is exactly the ToolFlood / runaway-worker defense.** If `worker-A` crashes and its TTL expires, `worker-B` can acquire — but `worker-A` coming back from a long GC pause **cannot** delete `worker-B`'s lock. Each worker only owns its own lock.


## 5. The mock agent handler

Before wiring the real LangGraph app, we'll use a **mock agent** for the queue/lock/store demo. Mock takes 1 second and emits 3 progress events, deterministic — so the pipeline mechanics are visible without 60s of LLM calls.

In [ ]:
import asyncio

from deepbrief.pipeline.progress import ProgressEmitter


async def mock_agent(task: dict, emitter: ProgressEmitter) -> dict:
    """Fake research run — emits 3 progress events, returns a stub draft."""
    topic = task["topic"]

    await emitter.emit(step=1, kind="thought", text=f"decomposing: {topic}")
    await asyncio.sleep(0.3)

    await emitter.emit(step=2, kind="tool_call", tool="web_search", query=topic)
    await asyncio.sleep(0.3)

    await emitter.emit(step=3, kind="thought", text="synthesizing draft")
    await asyncio.sleep(0.3)

    return {
        "answer": f"# Brief: {topic}\n\nMock draft for the pipeline demo.",
        "draft": f"# Brief: {topic}\n\nMock draft for the pipeline demo.",
    }

print("mock_agent defined — emits 3 events, returns in ~1s")

## 6. The six-step worker loop

Now the heart of S9 §5.4. Read the source. **The order of these steps is the interview answer.**

In [ ]:
from deepbrief.pipeline.worker import make_handler
from deepbrief.pipeline.store import ResultStore
from deepbrief.pipeline.streams import enqueue, consume_loop

# Pull out just the handler body — read it carefully
src = inspect.getsource(make_handler)
print(src)

**The order that matters** (S9 §5.5):

| Step | Why this order |
|---|---|
| 1. consume | obvious — can't act without a message |
| 2. existing-result check | short-circuits replays of completed work without re-running |
| 3. acquire lock | prevents two live workers running the same task |
| 4. run agent | the expensive part — protected by lock + completion check |
| 5. **persist** | **before ack** — if we ack first then crash before storing, the message is gone but no result exists. Silent data loss. |
| 6. **ack** | only after successful persist — if we crash here, the message redelivers and step 2 catches it |
| 7. release lock (finally) | **after ack** — if we release first and crash before ack, redelivery goes to a fresh lock and we re-run. By releasing last the new worker either finds the existing result or waits for TTL |


## 7. Run a worker, enqueue some tasks

Start the worker as an asyncio task. Enqueue three tasks. Watch them flow through.

In [ ]:
from deepbrief.pipeline.worker import run_worker

store = ResultStore("./data/runs.db")

# Kick off the worker — it'll process up to 3 messages then return
worker_task = asyncio.create_task(
    run_worker(
        redis=redis, store=store,
        agent_handler=mock_agent,
        group="deepbrief-nb09",
        stop_after=3,
    )
)

# Enqueue 3 tasks
for topic in ["MCP in 2026", "A2A vs MCP", "Tool-RAG patterns"]:
    msg_id = await enqueue(redis, {"topic": topic, "query": topic})
    print(f"enqueued: {topic} → msg={msg_id}")

# Wait for the worker to drain
n_processed = await worker_task
print(f"\nworker processed {n_processed} tasks")

## 8. Verify the SQLite result store

In [ ]:
# Query all stored runs
import sqlite3
rows = sqlite3.connect("./data/runs.db").execute(
    "SELECT task_id, query, substr(answer, 1, 60) AS preview FROM runs"
).fetchall()
for r in rows:
    print(r)

## 9. Idempotency demo — enqueue the same task_id twice

Real queues at-least-once-deliver. We simulate a redelivery: enqueue the **same** `task_id` twice, watch the worker do the work once and short-circuit the second time at step 2 (existing-result check).

In [ ]:
# Fresh worker for this demo, processes up to 2 messages
worker_task = asyncio.create_task(
    run_worker(
        redis=redis, store=store, agent_handler=mock_agent,
        group="deepbrief-nb09", stop_after=2,
    )
)

# Same task_id twice — second is the simulated redelivery
SHARED_ID = "dedupe-demo-001"
await enqueue(redis, {"topic": "WebGPU adoption", "task_id": SHARED_ID})
await enqueue(redis, {"topic": "WebGPU adoption", "task_id": SHARED_ID})

await worker_task

# How many DB rows? Should be exactly 1.
count = sqlite3.connect("./data/runs.db").execute(
    "SELECT COUNT(*) FROM runs WHERE task_id = ?", (SHARED_ID,)
).fetchone()[0]
print(f"rows with task_id={SHARED_ID}: {count}")
assert count == 1, "dedupe failed — should have exactly one row"
print("✅ dedupe worked: second delivery short-circuited at the existing-result check")

**Two layers of idempotency, both needed:**

- **In-flight protection** — SETNX lock prevents two workers running concurrently
- **Completed-task protection** — the existing-result check catches replays of finished work

Drop either and you get either double-execution (no lock) or wasted retries that re-run completed work (no completion check).


## 10. Progress streaming — what the SSE endpoint would see

The FastAPI SSE endpoint (`src/deepbrief/pipeline/api.py::stream_progress`) just subscribes to the same Pub/Sub channel. We can do the same from the notebook — start a subscriber, kick off a task, watch the events arrive in real time.

In [ ]:
from deepbrief.pipeline.progress import subscribe

PROGRESS_TASK_ID = "progress-demo-001"

async def collect_progress(task_id, timeout=10):
    """Subscribe to progress events for task_id, stop on `final` or timeout."""
    events = []
    try:
        async with asyncio.timeout(timeout):
            async for ev in subscribe(redis, task_id):
                events.append(ev)
                if ev.get("kind") == "final":
                    break
    except asyncio.TimeoutError:
        pass
    return events

# Start the worker, subscriber, and producer concurrently
worker = asyncio.create_task(
    run_worker(redis=redis, store=store, agent_handler=mock_agent,
               group="deepbrief-nb09", stop_after=1)
)
subscriber = asyncio.create_task(collect_progress(PROGRESS_TASK_ID))
await asyncio.sleep(0.2)   # give the subscriber a moment to attach

await enqueue(redis, {"topic": "streaming demo", "task_id": PROGRESS_TASK_ID})

events = await subscriber
await worker

print(f"received {len(events)} progress events:")
for ev in events:
    print(f"  step={ev.get('step'):>3}  kind={ev.get('kind'):12s}  ts={ev.get('ts'):.2f}")

**This is exactly what an SSE-subscribing browser would see.** The FastAPI endpoint in `api.py::stream_progress` wraps `subscribe()` in `EventSourceResponse` so the browser receives a continuous stream. Disconnect handling is automatic — the subscriber's `finally` block cleans up when the async generator is garbage-collected.


## 11. (Optional) Wire in the real LangGraph brief

Everything we've shown works with any async agent handler. The real version: plug in the LangGraph app from notebook 07b and run a real research task end-to-end.

This takes 30-60 seconds (real LLM + web search). Skip if you don't have a Tavily key.


In [ ]:
from deepbrief.agents.langgraph_brief import build_brief_app, ResearchState

brief_app = build_brief_app()

async def langgraph_agent(task: dict, emitter: ProgressEmitter) -> dict:
    """Real agent handler — runs the LangGraph brief and streams events from astream_events."""
    step = 0
    initial: ResearchState = {
        "topic": task["topic"], "sub_questions": [], "findings": [], "draft": None,
    }
    final_state = None
    async for ev in brief_app.astream_events(initial, version="v2"):
        kind = ev["event"]
        if kind == "on_chain_start" and ev.get("name") in {"decompose", "research", "synthesize"}:
            step += 1
            await emitter.emit(step=step, kind="node_start", node=ev["name"])
        if kind == "on_chain_end" and ev.get("name") == "synthesize":
            final_state = ev["data"]["output"]

    return {
        "answer": final_state.get("draft") if final_state else None,
        "draft": final_state.get("draft") if final_state else None,
    }

if os.getenv("TAVILY_API_KEY") or os.getenv("FORCE_REAL_AGENT"):
    worker = asyncio.create_task(
        run_worker(redis=redis, store=store, agent_handler=langgraph_agent,
                   group="deepbrief-nb09", stop_after=1)
    )
    await enqueue(redis, {"topic": "What is MCP?", "task_id": "real-1"})
    await worker
    rec = await store.find_one("real-1")
    print("=== real LangGraph run via pipeline ===")
    print(rec["answer"][:500] if rec else "<not stored>")
else:
    print("skipping real run — set TAVILY_API_KEY to enable")

## 12. What you've built

Look back at the lecture S9 §5.1 diagram. You now have **every box**:

- ✅ Producer → enqueue + a FastAPI app at `api.py`
- ✅ Queue → Redis Stream `agent:runs`, durable + consumer-group aware
- ✅ Worker — six-step loop with correct ordering
- ✅ Idempotency lock — SETNX claim + Lua release
- ✅ Progress emission — Pub/Sub channel per task
- ✅ Result store — SQLite (one class swap for MongoDB / Postgres)
- ✅ SSE endpoint — `api.py::stream_progress`

**This is ~250 lines of Python.** A production team would add: backoff on transient errors, dead-letter queue for permanent failures (XPENDING + XCLAIM), Prometheus metrics, distributed tracing via OpenTelemetry GenAI conventions. Each is a known add-on — none change the shape of what you just built.


## 13. Cleanup

In [ ]:
await redis.delete(STREAM)
for key in await redis.keys("agent:lock:*"):
    await redis.delete(key)
await redis.close()
store.close()
print("closed Redis + sqlite")

## 14. Self-check (senior-grade)

1. Why does step 5 (persist) come before step 6 (ack), not after? Walk through the silent-data-loss scenario if you flip them.
2. Why does the lock release happen in `finally` (after ack), and not before ack? Walk through the redelivery scenario if you flip them.
3. What end-to-end guarantees give you idempotency? **Two** layers — name both.
4. What does the Lua release script do that a plain `GET` + `if owner == me: DEL` doesn't?
5. The handler returns `False` on exception (no ack). What kicks in to prevent the next delivery from re-running the same task immediately?
6. Which of these would you keep if you migrated from Redis Streams to Kafka? Which would you change?

## What's next

You've finished the S9 hands-on track. Three places this goes next as portfolio work:

- **Dead-letter queue** — when a task fails 3 times, XCLAIM it to a `dead_letter` group; surface to ops.
- **Backpressure** — track stream length; refuse new enqueues above a threshold.
- **Trace integration** — emit OpenTelemetry GenAI spans (S8 §7.2) so the full request-trace shows worker latency, LLM latency, tool latency.

Each is ~half a day of work on top of what you have.
